This version:

- makes summaries **shorter and denser**, closer to dev-style plot summaries
- reduces the number of **easy standard triples**
- increases the proportion of **hard negatives** and **aspect-controlled** triples
- forces negatives to remain **surface-similar** to the anchor
- rejects outputs that are **too long**, **too copied**, or **too lexically easy**
- adds an optional **LLM judge / verifier** step
- writes accepted triples incrementally to JSONL and can **resume after interruption**


In [ ]:
import json
import os
import random
import re
import sys
import time
import urllib.request
from pathlib import Path
from typing import Optional, Tuple, Dict, Any
from collections import Counter

import numpy as np



OUTPUT_PATH = "/kaggle/working/synthetic_data_new_dev_like.jsonl"
CACHE_PATH  = OUTPUT_PATH   # output file also acts as resume cache

DEV_PATH    = "/kaggle/input/datasets/similarity/dev_track_a.jsonl"

# Backend
BACKEND      = "ollama"             
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"
API_MODEL    = "claude-sonnet-4-20250514"

# More dev-like allocation
N_STANDARD   = 300
N_ASPECT     = 1000   # 250 per sub-pattern
N_HARD_NEG   = 600
N_TOTAL      = N_STANDARD + N_ASPECT + N_HARD_NEG

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_RETRIES_PER_TRIPLE = 5
USE_LLM_JUDGE = True

# Length constraints will be derived from dev, but these are safe fallbacks
MIN_WORDS = 80
MAX_WORDS = 150
MIN_SENTS = 4
MAX_SENTS = 7

# Lexical / copying filters
MAX_COPIED_5GRAM_RATIO_POS = 0.20
MAX_COPIED_5GRAM_RATIO_NEG = 0.18
MAX_TOKEN_JACCARD_POS      = 0.45
MIN_TOKEN_JACCARD_NEG      = 0.05
MAX_TOKEN_JACCARD_NEG      = 0.35
MAX_EASY_MARGIN            = 0.12   # keep lexical gap between closer and farther relatively small

GENRES = [
    "crime thriller", "romantic comedy", "war drama", "science fiction",
    "heist film", "political drama", "coming-of-age", "psychological thriller",
    "adventure film", "supernatural horror", "legal drama", "espionage thriller",
    "road movie", "survival story", "family drama", "sports drama",
    "historical epic", "detective mystery", "redemption story", "revenge thriller",
    "courtroom drama", "dystopian fiction", "western", "medical drama",
    "conspiracy thriller",
]

ARCHETYPES = [
    "a protagonist seeks revenge for a loved one's death",
    "an outsider must prove themselves worthy to join a group",
    "a character discovers a dark secret about their own family",
    "two enemies are forced to cooperate in order to survive",
    "a protagonist must choose between duty and personal loyalty",
    "an underdog competes against a powerful corrupt establishment",
    "a character tries to escape and leave behind their past identity",
    "a mentor and protégé relationship is shattered by betrayal",
    "an investigator slowly realizes they are part of the conspiracy they are uncovering",
    "a protagonist sacrifices everything for the sake of others",
    "a false accusation forces a character to prove their innocence",
    "a character is gradually manipulated and begins to lose touch with reality",
    "strangers from opposing worlds form an unlikely bond during a shared crisis",
    "an obsessive pursuit of a single goal ultimately destroys the pursuer",
    "a character returns home after a long absence to find everything has changed",
    "a person of privilege confronts the consequences of their past actions",
    "two people compete for the same goal but fall in love along the way",
    "a whistleblower exposes a powerful institution at great personal cost",
    "a reformed criminal is pulled back into their old life by circumstances",
    "a character must complete one final mission before they can be free",
]


In [ ]:
# DEV STATS

def load_jsonl(path):
    rows = []
    p = Path(path)
    if not p.exists():
        print(f"[WARN] File not found: {path}")
        return rows
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+(?:[-']\w+)?\b", text or ""))

def sent_count(text: str) -> int:
    text = (text or "").strip()
    if not text:
        return 0
    parts = [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    return len(parts)

def describe_lengths(rows):
    texts = []
    for r in rows:
        for field in ["anchor_text", "text_a", "text_b"]:
            if r.get(field):
                texts.append(r[field])

    wc = np.array([word_count(t) for t in texts]) if texts else np.array([MIN_WORDS, MAX_WORDS])
    sc = np.array([sent_count(t) for t in texts]) if texts else np.array([MIN_SENTS, MAX_SENTS])

    stats = {
        "word_p10": int(np.percentile(wc, 10)),
        "word_p50": int(np.percentile(wc, 50)),
        "word_p90": int(np.percentile(wc, 90)),
        "sent_p10": int(np.percentile(sc, 10)),
        "sent_p50": int(np.percentile(sc, 50)),
        "sent_p90": int(np.percentile(sc, 90)),
    }
    return stats

DEV_ROWS = load_jsonl(DEV_PATH)
DEV_STATS = describe_lengths(DEV_ROWS)

TARGET_MIN_WORDS = max(MIN_WORDS, DEV_STATS["word_p10"])
TARGET_MAX_WORDS = min(MAX_WORDS, DEV_STATS["word_p90"])
TARGET_MIN_SENTS = max(MIN_SENTS, DEV_STATS["sent_p10"])
TARGET_MAX_SENTS = min(MAX_SENTS, DEV_STATS["sent_p90"])

print("Loaded dev rows:", len(DEV_ROWS))
print("DEV_STATS:", DEV_STATS)
print("Target words:", TARGET_MIN_WORDS, "to", TARGET_MAX_WORDS)
print("Target sentences:", TARGET_MIN_SENTS, "to", TARGET_MAX_SENTS)


## Prompting strategy

- **shorter** and more compressed
- **less decorative**
- explicit about **surface changes**
- explicit about **maintaining or changing only one narrative aspect**
- hard negatives are required to stay **surface-near** while becoming **structurally different**


In [ ]:
# PROMPTS

SYSTEM = (
    "You are a writing assistant creating Wikipedia-style film plot summaries. "
    "Write compact, neutral, third-person plot summaries with clear causal structure. "
    "Do not include a title, year, commentary, or bullet points. "
    "Keep each summary concise and natural, usually 4 to 6 sentences and about 90 to 140 words."
)

SEED_PROMPT = """\
Write a compact story summary in the style of a Wikipedia film plot section.

Requirements:
- Genre: {genre}
- Central situation: {archetype}
- 4 to 6 sentences
- About 90 to 140 words
- One clear causal chain
- Neutral encyclopedic tone
- No title, year, or commentary

The summary should feel like a human-written dev-set example: concise, plot-focused, and not overly elaborate.
Begin immediately with the summary."""

SIMILAR_PROMPT = """\
Here is an anchor story:

{anchor}

Write a new story that is narratively close to the anchor:
- preserve the same main causal chain
- preserve the same overall outcome
- change all names, specific places, and most surface details
- do NOT copy phrases from the anchor
- keep the genre/register compatible
- 4 to 6 sentences
- 90 to 140 words

The result should be structurally similar, but not a near-paraphrase.
Begin immediately with the summary."""

DISTRACTOR_SURFACE_SIMILAR_PROMPT = """\
Here is an anchor story:

{anchor}

Write a new story that stays superficially close to the anchor:
- same genre or neighboring genre
- similar protagonist role or social position
- similar setting type or atmosphere
- some shared topical cues

But make the narrative structure genuinely different:
- different central conflict
- different causal chain
- different resolution/outcome

This should be a hard distractor: surface-similar, structurally different.
Do NOT copy phrases from the anchor.
Write 4 to 6 sentences, about 90 to 140 words.
Begin immediately with the summary."""

SAME_COA_DIFF_OUTCOME = """\
Here is an anchor story:

{anchor}

Write a new story that keeps the same core course of action:
- the same type of events happen in roughly the same order
- the same plan or pursuit drives the plot

But change the final outcome in a meaningful way:
- if the anchor ends in success, make this end in failure or loss
- if the anchor ends in reunion, make this end in separation
- if the anchor ends in survival, make this end in death or permanent damage

Also:
- change names, places, and surface details
- do NOT copy wording
- keep it 4 to 6 sentences, about 90 to 140 words

Begin immediately with the summary."""

DIFF_COA_SAME_OUTCOME = """\
Here is an anchor story:

{anchor}

Write a new story that reaches the same type of final outcome as the anchor,
but through a clearly different course of action:
- different events
- different obstacles
- different route to the resolution
- same final state in abstract terms

Also:
- change names, places, and surface details
- do NOT copy wording
- keep it 4 to 6 sentences, about 90 to 140 words

Begin immediately with the summary."""

SAME_COA_SAME_OUTCOME = """\
Here is an anchor story:

{anchor}

Write a new story that remains close to the anchor in both:
- course of action
- final outcome

But still make it a distinct story:
- change names, places, institutions, and concrete details
- avoid phrase copying
- keep only the abstract narrative structure

Write 4 to 6 sentences, about 90 to 140 words.
Begin immediately with the summary."""

DIFF_COA_DIFF_OUTCOME = """\
Here is an anchor story:

{anchor}

Write a story that is thematically related to the anchor and superficially compatible with it,
but differs in both:
- course of action
- final outcome

It should feel like a plausible distractor for a narrative similarity task:
surface-near, structurally different.

Do not copy phrases. Write 4 to 6 sentences, about 90 to 140 words.
Begin immediately with the summary."""

JUDGE_PROMPT = """\
You are validating a narrative similarity training triple.

Anchor:
{anchor}

Candidate A:
{a}

Candidate B:
{b}

Return strict JSON only with this schema:
{{
  "closer": "A" or "B",
  "difficulty": "easy" or "medium" or "hard",
  "same_coa_a": true/false,
  "same_coa_b": true/false,
  "same_outcome_a": true/false,
  "same_outcome_b": true/false,
  "same_theme_a": true/false,
  "same_theme_b": true/false,
  "copied_phrase_problem": true/false,
  "reason": "one short sentence"
}}

Important:
- "closer" means closer in overall narrative structure to the anchor
- copied_phrase_problem should be true if one candidate is mostly a surface paraphrase
- prefer "hard" when both candidates are superficially plausible
Return JSON only.
"""


In [ ]:
# BACKEND + TEXT CLEANING

def _ollama_available():
    try:
        urllib.request.urlopen(OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False

def _anthropic_available():
    return bool(os.environ.get("ANTHROPIC_API_KEY", ""))

def _generate_ollama(prompt: str, max_tokens: int = 320) -> str:
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": SYSTEM + "\n\n" + prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens,
            "temperature": 0.8,
            "top_p": 0.95,
            "repeat_penalty": 1.12,
        },
    }).encode()

    req = urllib.request.Request(
        OLLAMA_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read()).get("response", "").strip()

def _generate_api(prompt: str, max_tokens: int = 320) -> str:
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")

    payload = json.dumps({
        "model": API_MODEL,
        "max_tokens": max_tokens,
        "system": SYSTEM,
        "messages": [{"role": "user", "content": prompt}],
    }).encode()

    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data=payload,
        headers={
            "Content-Type": "application/json",
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    return data["content"][0]["text"].strip()

def generate(prompt: str, max_tokens: int = 320) -> str:
    for attempt in range(3):
        try:
            if BACKEND == "ollama":
                return _generate_ollama(prompt, max_tokens=max_tokens)
            return _generate_api(prompt, max_tokens=max_tokens)
        except Exception as e:
            if attempt == 2:
                print(f"[ERROR] generation failed: {e}")
                return ""
            time.sleep(2 ** attempt)
    return ""

def _clean(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(
        r"^(?:here(?:'s| is)|sure[!,]?|certainly[!,]?)(?:[^:\n]*)(?::\s*|\n+)",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    text = re.sub(r'^\*\*[^*]+\*\*\s*\n+', '', text)
    text = re.sub(r'^(?:story|summary|plot)\s*:\s*', '', text, flags=re.IGNORECASE)
    return text.strip()

def _extract_first_json(text: str) -> Optional[dict]:
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


In [ ]:
# HEURISTICS / VALIDATION

def _tokens(text: str):
    return re.findall(r"\b\w+(?:[-']\w+)?\b", (text or "").lower())

def token_jaccard(a: str, b: str) -> float:
    A, B = set(_tokens(a)), set(_tokens(b))
    if not A and not B:
        return 0.0
    return len(A & B) / max(1, len(A | B))

def ngrams(tokens, n=5):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def copied_ngram_ratio(a: str, b: str, n: int = 5) -> float:
    A = ngrams(_tokens(a), n=n)
    B = ngrams(_tokens(b), n=n)
    if not B:
        return 0.0
    return len(A & B) / len(B)

def length_ok(text: str) -> bool:
    wc = word_count(text)
    sc = sent_count(text)
    return TARGET_MIN_WORDS <= wc <= TARGET_MAX_WORDS and TARGET_MIN_SENTS <= sc <= TARGET_MAX_SENTS

def valid_summary(text: str) -> bool:
    return bool(text) and length_ok(text)

def judge_triple(anchor: str, a: str, b: str) -> Optional[dict]:
    if not USE_LLM_JUDGE:
        return None
    raw = generate(JUDGE_PROMPT.format(anchor=anchor, a=a, b=b), max_tokens=220)
    parsed = _extract_first_json(raw)
    return parsed if isinstance(parsed, dict) else None

def validate_triple(anchor: str, closer: str, farther: str, generation_type: str) -> Tuple[bool, dict]:
    meta = {
        "anchor_words": word_count(anchor),
        "closer_words": word_count(closer),
        "farther_words": word_count(farther),
        "anchor_sents": sent_count(anchor),
        "closer_sents": sent_count(closer),
        "farther_sents": sent_count(farther),
        "closer_jaccard": token_jaccard(anchor, closer),
        "farther_jaccard": token_jaccard(anchor, farther),
        "closer_copy5": copied_ngram_ratio(anchor, closer, n=5),
        "farther_copy5": copied_ngram_ratio(anchor, farther, n=5),
    }

    if not valid_summary(anchor) or not valid_summary(closer) or not valid_summary(farther):
        meta["reject_reason"] = "length_out_of_range"
        return False, meta

    if meta["closer_copy5"] > MAX_COPIED_5GRAM_RATIO_POS:
        meta["reject_reason"] = "closer_too_copied"
        return False, meta

    if meta["farther_copy5"] > MAX_COPIED_5GRAM_RATIO_NEG:
        meta["reject_reason"] = "farther_too_copied"
        return False, meta

    if meta["closer_jaccard"] > MAX_TOKEN_JACCARD_POS:
        meta["reject_reason"] = "closer_surface_too_high"
        return False, meta

    if meta["farther_jaccard"] < MIN_TOKEN_JACCARD_NEG:
        meta["reject_reason"] = "farther_surface_too_low"
        return False, meta

    if meta["farther_jaccard"] > MAX_TOKEN_JACCARD_NEG:
        meta["reject_reason"] = "farther_surface_too_high"
        return False, meta

    lexical_margin = meta["closer_jaccard"] - meta["farther_jaccard"]
    meta["lexical_margin"] = lexical_margin

    # Reject overly easy positives where lexical overlap alone gives the answer
    if lexical_margin > MAX_EASY_MARGIN and generation_type != "standard":
        meta["reject_reason"] = "too_lexically_easy"
        return False, meta

    judge = judge_triple(anchor, closer, farther)
    meta["judge"] = judge

    if judge is not None:
        if judge.get("closer") != "A":
            meta["reject_reason"] = "judge_disagrees"
            return False, meta

        if judge.get("copied_phrase_problem") is True:
            meta["reject_reason"] = "judge_copy_problem"
            return False, meta

        # For dev-like data, we prefer medium/hard, except some standard examples
        if generation_type != "standard" and judge.get("difficulty") == "easy":
            meta["reject_reason"] = "judge_too_easy"
            return False, meta

    meta["reject_reason"] = None
    return True, meta


In [ ]:
# TRIPLE BUILDERS

def build_anchor(genre: str, archetype: str) -> str:
    return _clean(generate(SEED_PROMPT.format(genre=genre, archetype=archetype)))

def build_standard_triple(genre: str, archetype: str, model_name: str) -> Optional[dict]:
    for _ in range(MAX_RETRIES_PER_TRIPLE):
        anchor  = build_anchor(genre, archetype)
        closer  = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))
        farther = _clean(generate(DISTRACTOR_SURFACE_SIMILAR_PROMPT.format(anchor=anchor)))

        ok, meta = validate_triple(anchor, closer, farther, "standard")
        if not ok:
            continue

        if random.random() < 0.5:
            text_a, text_b, label = closer, farther, True
        else:
            text_a, text_b, label = farther, closer, False

        return {
            "model": model_name,
            "anchor_text": anchor,
            "text_a": text_a,
            "text_b": text_b,
            "text_a_is_closer": label,
            "generation_type": "standard",
            "seed_genre": genre,
            "seed_archetype": archetype,
            "validation": meta,
        }
    return None

def build_aspect_triple(genre: str, archetype: str, pattern: str, model_name: str) -> Optional[dict]:
    prompt_map = {
        "same_coa_diff_outcome": SAME_COA_DIFF_OUTCOME,
        "diff_coa_same_outcome": DIFF_COA_SAME_OUTCOME,
        "same_coa_same_outcome": SAME_COA_SAME_OUTCOME,
        "diff_coa_diff_outcome": DIFF_COA_DIFF_OUTCOME,
    }

    for _ in range(MAX_RETRIES_PER_TRIPLE):
        anchor = build_anchor(genre, archetype)

        if pattern in ("same_coa_diff_outcome", "diff_coa_same_outcome", "same_coa_same_outcome"):
            closer  = _clean(generate(prompt_map[pattern].format(anchor=anchor)))
            farther = _clean(generate(DISTRACTOR_SURFACE_SIMILAR_PROMPT.format(anchor=anchor)))
        else:
            closer  = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))
            farther = _clean(generate(prompt_map[pattern].format(anchor=anchor)))

        ok, meta = validate_triple(anchor, closer, farther, f"aspect_{pattern}")
        if not ok:
            continue

        if random.random() < 0.5:
            text_a, text_b, label = closer, farther, True
        else:
            text_a, text_b, label = farther, closer, False

        return {
            "model": model_name,
            "anchor_text": anchor,
            "text_a": text_a,
            "text_b": text_b,
            "text_a_is_closer": label,
            "generation_type": f"aspect_{pattern}",
            "seed_genre": genre,
            "seed_archetype": archetype,
            "validation": meta,
        }
    return None

def build_hard_negative_triple(genre: str, archetype: str, model_name: str) -> Optional[dict]:
    for _ in range(MAX_RETRIES_PER_TRIPLE):
        anchor  = build_anchor(genre, archetype)
        closer  = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))
        farther = _clean(generate(DISTRACTOR_SURFACE_SIMILAR_PROMPT.format(anchor=anchor)))

        ok, meta = validate_triple(anchor, closer, farther, "hard_negative")
        if not ok:
            continue

        # For hard negatives we still need one candidate to be closer,
        # but we want the pair to remain hard.
        if random.random() < 0.5:
            text_a, text_b, label = closer, farther, True
        else:
            text_a, text_b, label = farther, closer, False

        return {
            "model": model_name,
            "anchor_text": anchor,
            "text_a": text_a,
            "text_b": text_b,
            "text_a_is_closer": label,
            "generation_type": "hard_negative",
            "seed_genre": genre,
            "seed_archetype": archetype,
            "validation": meta,
        }
    return None


In [ ]:
# RESUME / PLAN

def load_cache(path: str) -> set:
    p = Path(path)
    if not p.exists():
        return set()
    done = set()
    with open(p, encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done.add((obj["seed_genre"], obj["seed_archetype"], obj["generation_type"]))
            except Exception:
                pass
    return done

def append_result(path: str, result: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

def build_plan(rng: random.Random) -> list:
    aspect_patterns = [
        "same_coa_diff_outcome",
        "diff_coa_same_outcome",
        "same_coa_same_outcome",
        "diff_coa_diff_outcome",
    ]
    n_per_aspect = N_ASPECT // len(aspect_patterns)

    tasks = []

    for _ in range(N_STANDARD):
        tasks.append({
            "genre": rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type": "standard",
        })

    for p in aspect_patterns:
        for _ in range(n_per_aspect):
            tasks.append({
                "genre": rng.choice(GENRES),
                "archetype": rng.choice(ARCHETYPES),
                "type": f"aspect_{p}",
            })

    for _ in range(N_HARD_NEG):
        tasks.append({
            "genre": rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type": "hard_negative",
        })

    rng.shuffle(tasks)
    return tasks


In [ ]:
def main():
    if BACKEND == "ollama":
        model_tag = OLLAMA_MODEL
        if not _ollama_available():
            print("ERROR: Ollama is not available.")
            print("Run: ollama serve && ollama pull", OLLAMA_MODEL)
            return
    else:
        model_tag = API_MODEL
        if not _anthropic_available():
            print("ERROR: ANTHROPIC_API_KEY not set.")
            return

    print("Backend:", BACKEND, "| model:", model_tag)
    print("Output:", OUTPUT_PATH)
    print("Target total:", N_TOTAL)
    print("Mix:",
          {"standard": N_STANDARD, "aspect": N_ASPECT, "hard_negative": N_HARD_NEG})
    print("Dev-like length target:", TARGET_MIN_WORDS, "-", TARGET_MAX_WORDS, "words |",
          TARGET_MIN_SENTS, "-", TARGET_MAX_SENTS, "sentences")
    print()

    rng = random.Random(SEED)
    plan = build_plan(rng)
    done_keys = load_cache(CACHE_PATH)

    print("Already generated:", len(done_keys))
    print()

    counts = Counter()
    rejects = Counter()
    errors = 0
    start = time.time()

    for i, task in enumerate(plan, start=1):
        triple_type = task["type"]
        genre = task["genre"]
        archetype = task["archetype"]

        cache_key = (genre, archetype, triple_type)
        if cache_key in done_keys:
            continue

        try:
            if triple_type == "standard":
                result = build_standard_triple(genre, archetype, model_tag)
            elif triple_type.startswith("aspect_"):
                pattern = triple_type[len("aspect_"):]
                result = build_aspect_triple(genre, archetype, pattern, model_tag)
            else:
                result = build_hard_negative_triple(genre, archetype, model_tag)

            if result is None:
                errors += 1
                rejects["exhausted_retries"] += 1
                continue

            append_result(OUTPUT_PATH, result)
            done_keys.add(cache_key)
            counts[result["generation_type"]] += 1

        except Exception as e:
            errors += 1
            print(f"[ERROR] {triple_type} | {genre} | {archetype} -> {e}")
            continue

        total_done = sum(counts.values())
        if total_done > 0 and total_done % 25 == 0:
            elapsed = time.time() - start
            rate = total_done / max(elapsed, 1e-9)
            print(f"[{total_done}] {rate:.3f} triples/s | errors={errors}")
            print("Counts:", dict(counts))
            print()

    elapsed = time.time() - start
    total = sum(counts.values())

    print("=" * 60)
    print(f"Done in {elapsed/60:.1f} min")
    print("Accepted:", total)
    print("Errors / exhausted:", errors)
    print("Counts:", dict(counts))
    print("Output:", OUTPUT_PATH)
    print("=" * 60)

main()


To get data **closer to dev quality**, the key idea is not just better prompts, but also **better rejection**.

This notebook rejects triples that are:

- too long or too short compared with dev
- too copied from the anchor
- too easy lexically
- judged as easy by the verifier

That does **not guarantee** dev-level quality, because the dev set is still human-curated and naturally ambiguous. But it does move the synthetic data much closer than the original generator.


In [ ]:
# Quick analysis of generated output after a run

def summarize_output(path=OUTPUT_PATH):
    p = Path(path)
    if not p.exists():
        print("No output file yet.")
        return

    rows = load_jsonl(path)
    print("Rows:", len(rows))

    gtypes = Counter(r["generation_type"] for r in rows)
    print("Generation types:", dict(gtypes))

    labels = np.mean([bool(r["text_a_is_closer"]) for r in rows]) * 100 if rows else 0.0
    print(f"text_a_is_closer=True: {labels:.2f}%")

    all_anchor = [r["anchor_text"] for r in rows]
    all_a = [r["text_a"] for r in rows]
    all_b = [r["text_b"] for r in rows]

    for name, texts in [("anchor", all_anchor), ("text_a", all_a), ("text_b", all_b)]:
        wc = np.array([word_count(t) for t in texts])
        sc = np.array([sent_count(t) for t in texts])
        print(f"{name}: words mean={wc.mean():.1f}, p10={np.percentile(wc,10):.1f}, p90={np.percentile(wc,90):.1f}")
        print(f"{name}: sents mean={sc.mean():.1f}, p10={np.percentile(sc,10):.1f}, p90={np.percentile(sc,90):.1f}")

summarize_output()
